# 01 · Exploratory Data Analysis

Understand the dataset structure, check distributions, and surface early attrition signals
before any modelling. Good EDA is what separates a portfolio project from a tutorial.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/employee_data.csv')
print(f"Shape: {df.shape}")
print(f"Attrition rate: {df['attrited'].mean():.1%}  ({df['attrited'].sum()} of {len(df):,})")
df.head()

In [ ]:
# Data quality check
print("=== NULL COUNTS ===")
print(df.isnull().sum())
print("\n=== DTYPES ===")
print(df.dtypes)

In [ ]:
# Attrition by department
dept_stats = (df.groupby('department')['attrited']
                .agg(['mean','count','sum'])
                .rename(columns={'mean':'attrition_rate','count':'headcount','sum':'attrited_n'})
                .sort_values('attrition_rate', ascending=False))
dept_stats['attrition_rate'] = dept_stats['attrition_rate'].map('{:.1%}'.format)
print(dept_stats)

In [ ]:
# Attrition by tenure band
bins   = [0,1,2,4,7,12,20,100]
labels = ['<1yr','1-2yr','2-4yr','4-7yr','7-12yr','12-20yr','20+yr']
df['tenure_band'] = pd.cut(df['tenure_years'], bins=bins, labels=labels, right=True)

print(df.groupby('tenure_band', observed=True)['attrited']
        .agg(['mean','count'])
        .rename(columns={'mean':'rate','count':'n'})
        .assign(rate=lambda x: x['rate'].map('{:.1%}'.format)))

In [ ]:
# Key continuous driver: compa ratio vs. attrition
retained = df[df['attrited']==0]['compa_ratio']
attrited = df[df['attrited']==1]['compa_ratio']
print(f"Median compa ratio — Retained: {retained.median():.3f} | Attrited: {attrited.median():.3f}")
print(f"% below market (<0.90) — Retained: {(retained<0.90).mean():.1%} | Attrited: {(attrited<0.90).mean():.1%}")

In [ ]:
# Engagement score vs. attrition
e_ret = df[df['attrited']==0]['engagement_score']
e_att = df[df['attrited']==1]['engagement_score']
print(f"Mean engagement — Retained: {e_ret.mean():.2f} | Attrited: {e_att.mean():.2f}")
print(f"(t-test p-value will be computed in 02_analysis)")

In [ ]:
# --- FIGURE: Attrition Rate by Department ---
# Run this cell to reproduce fig1_attrition_by_department.png
# (Full styled version saved during project build; this is the quick version)

dept_plot = (df.groupby('department')['attrited']
               .mean()
               .sort_values()
               .mul(100))

fig, ax = plt.subplots(figsize=(8,4.5))
colors = ['#3FB950' if v < 8 else '#E3B341' if v < 12 else '#FF7B72'
          for v in dept_plot.values]
dept_plot.plot(kind='barh', ax=ax, color=colors)
ax.axvline(df['attrited'].mean()*100, color='grey', linestyle='--', alpha=0.7)
ax.set_xlabel('Attrition Rate (%)')
ax.set_title('Attrition Rate by Department')
plt.tight_layout()
plt.savefig('../outputs/figures/fig1_attrition_by_department.png', dpi=150, bbox_inches='tight')
plt.show()